1. Сколько у нас пользователей, которые совершили покупку только один раз? 


2. Сколько заказов в месяц в среднем не доставляется по разным причинам (вывести детализацию по причинам)?


3. По каждому товару определить, в какой день недели товар чаще всего покупается.


4. Сколько у каждого из пользователей в среднем покупок в неделю (по месяцам)? Не стоит забывать, что внутри месяца может быть не целое количество недель. Например в Ноябре 2021 года 4,28 недели. И внутри метрики это нужно учесть.


5. Напиши функцию на python, позволяющую строить когортный анализ. В период с января по декабрь выяви когорту с самым высоким retention на 3й месяц. 


6. Построй RFM-кластеры для пользователей. Выведи для каждого кластера средние значения метрик R, F, M. (подробно опиши, как были построены метрики R, F, M).

In [1]:
import pandas as pd
import numpy as np

from dateutil import parser
from pandas import DataFrame

In [2]:
users = pd.read_csv('olist_customers_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')

In [3]:
users.head(2)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP


In [4]:
orders.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00


In [5]:
orders.order_status.unique()

array(['delivered', 'invoiced', 'shipped', 'processing', 'unavailable',
       'canceled', 'created', 'approved'], dtype=object)

In [6]:
items.head(2)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93


### 1. Сколько у нас пользователей, которые совершили покупку только один раз? 

Будем считать, что покупка совершена, если статус заказа - "delivered".

In [7]:
users.merge(
    orders.query('order_status == "delivered"'), 
    on='customer_id', 
    how='left'
).groupby(
    'customer_unique_id',
    as_index=False
).order_id.count().query('order_id == 1').customer_unique_id.count()

90557

### 2. Сколько заказов в месяц в среднем не доставляется по разным причинам (вывести детализацию по причинам)?

In [8]:
def cuter(val):
    return val[:7]

In [9]:
orders['month'] = orders.order_purchase_timestamp.apply(cuter)

In [10]:
task_1_var_1 = (
    orders
    .query('order_status in ["unavailable", "canceled"]')
    .groupby('order_status')
    .order_id
    .count()
)

In [11]:
months_count = orders.query('order_status in ["unavailable", "canceled"]').month.nunique()

In [12]:
task_1_var_1 / months_count

order_status
canceled       26.041667
unavailable    25.375000
Name: order_id, dtype: float64

In [13]:
task_1_var_2 = (
    orders
    .query('order_status in ["unavailable", "canceled"]')
    .groupby(['month', 'order_status'], as_index=False)
    .order_id
    .count()
    .groupby('order_status')
    .order_id
    .mean()
)

In [14]:
task_1_var_2

order_status
canceled       26.041667
unavailable    29.000000
Name: order_id, dtype: float64

### 3. По каждому товару определить, в какой день недели товар чаще всего покупается.

Будем считать, что покупка совершена, если статус заказа не входит в следующий список `'created', 'canceled', 'unavaible'`

In [15]:
bad_statuses = ['created', 'canceled', 'unavaible']

def get_weekday(val):
    return pd.to_datetime(val).dt.weekday + 1

def most_frequent(series):
    return series.value_counts().index[0]

In [16]:
filtered_orders = orders.query('order_status not in @bad_statuses')[['order_id', 'order_approved_at']]
filtered_orders['weekday'] = get_weekday(filtered_orders.order_approved_at)
filtered_orders = filtered_orders[['order_id', 'weekday']].dropna()

In [17]:
items.merge(
    filtered_orders, 
    on='order_id', 
    how='inner'
).groupby(
    'product_id',
    as_index=False
).agg({'weekday': most_frequent})

,product_id,weekday
0,00066f42aeeb9f3007548bb9d3f33c38,7.0
1,00088930e925c41fd95ebfe695fd2655,2.0
2,0009406fd7479715e4bef61dd91f2462,5.0
3,000b8f95fcb9e0096488278317764d19,5.0
4,000d9be29b5207b54e86aa1b1ac54872,2.0
...,...,...
32727,fff6177642830a9a94a0f2cba5e476d1,6.0
32728,fff81cc3158d2725c0655ab9ba0f712c,1.0
32729,fff9553ac224cec9d15d49f5a263411f,6.0
32730,fffdb2d0ec8d6a61f0a0a0db3f25b441,2.0


### 4. Сколько у каждого из пользователей в среднем покупок в неделю (по месяцам)? Не стоит забывать, что внутри месяца может быть не целое количество недель. Например в Ноябре 2021 года 4,28 недели. И внутри метрики это нужно учесть.

In [18]:
orders['month'] = pd.to_datetime(orders['order_approved_at']).dt.to_period('M')
orders_per_month = users.merge(
    orders.query('order_status == "delivered"')
).groupby(
    ['customer_unique_id', 'month'],
    as_index=False
).order_id.count(
).rename(
    columns={'order_id': 'orders_count'}
)
orders_per_month['days_in_month'] = orders_per_month.month.apply(lambda x: pd.Period(x).days_in_month)
orders_per_month['mean_per_weekday'] = orders_per_month.apply(lambda x: x.orders_count / x.days_in_month * 7, axis=1)
orders_per_month

,customer_unique_id,month,orders_count,days_in_month,mean_per_weekday
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05,1,31,0.225806
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05,1,31,0.225806
2,0000f46a3911fa3c0805444483337064,2017-03,1,31,0.225806
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10,1,31,0.225806
4,0004aac84e0df4da2b147fca70cf8255,2017-11,1,30,0.233333
...,...,...,...,...,...
95171,fffcf5a5ff07b0908bd4e2dbc735a684,2017-06,1,30,0.233333
95172,fffea47cd6d3cc0a88bd621562a9d061,2017-12,1,31,0.225806
95173,ffff371b4d645b6ecea244b27531430a,2017-02,1,28,0.250000
95174,ffff5962728ec6157033ef9805bacc48,2018-05,1,31,0.225806


### 5. Напиши функцию на python, позволяющую строить когортный анализ. В период с января по декабрь выяви когорту с самым высоким retention на 3й месяц. 

In [19]:
def cohort_analysis(data):
    first_purchase = data.groupby('customer_unique_id', as_index=False).month.min().rename(columns={'month':'cohort'})
    data = data.merge(first_purchase)
    data['retained'] = data.month == data.cohort + 3
    return data.groupby(
        ['customer_unique_id', 'cohort'], 
        as_index=False
    ).retained.apply(
        lambda x: 1 if sum(x) else 0
    ).groupby('cohort').retained.mean() * 100
    
max_retention = cohort_analysis(users.merge(orders.query('order_status == "delivered"')))
print(f'{max_retention.idxmax()} наблюдался самый высокий retention: {max_retention.max()}%')

2017-06 наблюдался самый высокий retention: 0.42706964520367935%


### 6. Построй RFM-кластеры для пользователей. Выведи для каждого кластера средние значения метрик R, F, M. (подробно опиши, как были построены метрики R, F, M).

Зафиксируем Frequency как суммарное количество покупок у пользователя за весь период данных

In [20]:
users_frequency = users[['customer_unique_id', 'customer_id']].merge(
    orders.query('order_status == "delivered"')[['customer_id', 'order_id']], 
    on='customer_id', 
    how='left'
).groupby(
    'customer_unique_id', 
    as_index=False
).order_id.count()

In [21]:
users_frequency['frequency_cluster'] = np.digitize(users_frequency.order_id, [0, 1, 2])

In [22]:
users_frequency = users_frequency.rename({'order_id': 'orders_count'})

In [23]:
users_frequency

,customer_unique_id,order_id,frequency_cluster
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2
2,0000f46a3911fa3c0805444483337064,1,2
3,0000f6ccb0745a6a4b88665a16c9f078,1,2
4,0004aac84e0df4da2b147fca70cf8255,1,2
...,...,...,...
96091,fffcf5a5ff07b0908bd4e2dbc735a684,1,2
96092,fffea47cd6d3cc0a88bd621562a9d061,1,2
96093,ffff371b4d645b6ecea244b27531430a,1,2
96094,ffff5962728ec6157033ef9805bacc48,1,2
